# Практика 3. Доходность и риск вашей акции
**Количественные финансы · НИУ ВШЭ · 19 ноября 2026**

Продолжаем работать с акцией, которая назначена вам в самостоятельной работе 2 ([таблица распределения](https://docs.google.com/spreadsheets/d/1jDm3lILAcKnvIehthge5Q3NfrHSzqmem/edit?usp=sharing)). Сегодня отвечаем на вопрос: **что получил инвестор в эту акцию и каким риском заплатил – по сравнению с рынком и денежным рынком?**

Материал – [глава 4 «Доходность и риск»](https://quantviews.github.io/quant-finance-book/04-returns-risk.html).

| Когда | Задания |
|---|---|
| Вторая пара · 60 минут | 0–4 и 7 |
| Дома | 5, 6, 8, 9 и всё, что не успели |

Сдаём выполненный ноутбук `practice03_фамилия.ipynb` с ответами в ячейках «**Ответ:**» способом, который объявит преподаватель.

Если застряли больше чем на 10 минут, запишите в ответ, что пробовали, текст ошибки и вопрос, и переходите к следующему заданию.

**Имя и группа:** …

**Моя акция:** …

## 0. Подготовка
Следующие две ячейки подключают библиотеки и загружают данные. Выполните их один раз, редактировать не нужно. Данные загружаются с GitHub; без сети ноутбук читает файл `data/prices_2015_2026.csv` рядом с собой.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.options.display.float_format = "{:.4f}".format
print("Библиотеки загружены")

In [ ]:
# Загрузка данных: сначала с GitHub, без сети – из файла рядом с ноутбуком.
import hashlib, io, urllib.request
from pathlib import Path

URL = "https://raw.githubusercontent.com/quantviews/quant-finance-book/main/lessons/03-returns-risk/data/prices_2015_2026.csv"
LOCAL = Path("data/prices_2015_2026.csv")
try:
    raw = urllib.request.urlopen(URL, timeout=20).read()
    print("Данные загружены с GitHub")
except OSError:
    raw = LOCAL.read_bytes()
    print("Нет доступа к GitHub, данные прочитаны из файла", LOCAL)
assert hashlib.sha256(raw).hexdigest() == "99ada86289b50fe4d6753c04ce4feec7d255894aeb34f31e01b8647c98699bcf", "Файл отличается от снимка занятия"
df = pd.read_csv(io.BytesIO(raw), parse_dates=["date"]).set_index("date")
print("Строк:", len(df), "· период:", df.index.min().date(), "–", df.index.max().date())

### Что в таблице
Дневные данные за 05.01.2015–16.11.2026, одна строка – один торговый день. Снимок выгружен 17.11.2026; числа могут немного отличаться от примеров книги.

- `IMOEX` – индекс Мосбиржи, только цены. `MCFTR` – индекс полной доходности: те же акции, дивиденды реинвестируются.
- `RUONIA` – расчётный индекс денежного рынка по ставке RUONIA Банка России, 100 в первый день. Это ориентир безрисковой доходности, а не конкретный вклад.
- `SBER`, `GAZP`, … – `adj_close`: цены с поправкой на сплиты и дивиденды.
- `SBER_close`, `GAZP_close`, … – `close`: цены с поправкой на сплиты, без дивидендов.

Пустая ячейка означает, что в этот день сделок с бумагой не было или она ещё (уже) не торговалась. У MCFTR нет значения за 24.03.2022: биржа его не опубликовала. Источник: Московская биржа, Банк России; корректировка на дивиденды – расчёт автора курса.

In [ ]:
TICKER = "..."   # ваш тикер, например "GAZP"
assert TICKER in df.columns, "Такого тикера нет в таблице – проверьте написание"
px = df[TICKER]                   # adj_close
px_close = df[TICKER + "_close"]  # close
df[[TICKER, TICKER + "_close", "IMOEX", "MCFTR", "RUONIA"]].tail()

## Задание 1. Паспорт ряда · 10 минут
Прежде чем считать доходности, проверьте ряд своей акции.

1. Первая и последняя даты с ценой, число дней с ценой.
2. Сколько дней в календаре таблицы у вашей акции нет цены и когда они были? Есть ли они в середине ряда, а не только в начале и в конце?
3. Самый большой разрыв между соседними датами с ценой (в календарных днях) и между какими датами он был.

**Ответ:** …

In [ ]:
p = px.dropna()
print("С", p.index.min().date(), "по", p.index.max().date(), "· дней с ценой:", len(p))
missing = px.loc[p.index.min():p.index.max()].isna()
print("Пропусков внутри ряда:", missing.sum())
# Найдите самый большой разрыв: p.index.to_series().diff() – разница с предыдущей датой


## Задание 2. Две доходности · 10 минут
**Сначала предскажите:** в какой день за всю историю простая и логарифмическая доходности вашей акции различались сильнее всего? Это был обычный день?

1. Посчитайте простую `R` и логарифмическую `r` дневные доходности по `adj_close`.
2. Проверьте два тождества на всём ряду: сумма `r` равна `ln(последняя / первая цена)`; произведение `(1 + R)` минус 1 равно `последняя / первая − 1`.
3. Найдите день с наибольшей разницей `R − r` и объясните, почему разница большая именно там.

**Предсказание:** …

**Ответ:** …

In [ ]:
R = p.pct_change()            # простая доходность
r = np.log(p).diff()          # логарифмическая
print("Сумма r:", r.sum(), " ln(P_T/P_0):", np.log(p.iloc[-1] / p.iloc[0]))
# Допишите вторую проверку и поиск дня с наибольшей разницей R - r


## Задание 3. Что дали дивиденды · 15 минут
Среднегодовой темп роста (CAGR) = (последняя / первая цена)^(1 / число лет) − 1.

1. Посчитайте CAGR вашей акции по `close` и по `adj_close` за весь период. Какую часть результата дали дивиденды (в процентных пунктах в год)?
2. Сравните с IMOEX и MCFTR: на сколько п.п. в год различаются они?
3. Найдите три дня, в которые доходность по `adj_close` сильнее всего превышала доходность по `close`. Что это за дни? Проверьте один из них по истории дивидендов (например, на сайте [закрытияреестров.рф](https://закрытияреестров.рф) или у эмитента).

**Ответ:** …

In [ ]:
def cagr(series):
    s = series.dropna()
    years = (s.index[-1] - s.index[0]).days / 365.25
    return (s.iloc[-1] / s.iloc[0]) ** (1 / years) - 1

print(f"CAGR по adj_close: {cagr(px):.2%}")
# Посчитайте CAGR по close, для IMOEX и MCFTR; найдите три дня с наибольшей разницей доходностей


## Задание 4. Волатильность · 15 минут
1. Посчитайте годовую волатильность вашей акции, IMOEX и MCFTR по дневным лог-доходностям. Во сколько раз ваша акция волатильнее рынка?
2. Постройте на одном графике скользящую 63-дневную годовую волатильность вашей акции и IMOEX.
3. Найдите периоды, когда волатильность акции была более чем вдвое выше рыночной. С какими событиями в компании или отрасли это связано?

**Внимание к пропускам.** Считайте доходности так, чтобы дни без цены не превращались в нулевые доходности: `np.log(prices).diff()` или `pct_change(fill_method=None)`.

**Ответ:** …

In [ ]:
lr = np.log(df[[TICKER, "IMOEX", "MCFTR"]]).diff()
# годовая волатильность: дневное стандартное отклонение × √252
# скользящая: lr.rolling(63).std() * np.sqrt(252)


## Задание 5. Худшие дни и хвосты · дома, 15 минут
1. Найдите пять худших дней вашей акции по простой доходности `adj_close`.
2. Для каждого дня укажите: доходность IMOEX в тот же день; сколько календарных дней прошло с предыдущей даты с ценой; было ли это падение всего рынка, новость компании, дивидендная отсечка или разрыв в торгах.
3. Сколько дней с отклонением от среднего сильнее 4 стандартных отклонений у вашей акции? При нормальном распределении ожидалось бы `n × 0,0000633` таких дней, где `n` – число доходностей.

**Ответ:** …

In [ ]:
ret = df[[TICKER, "IMOEX"]].pct_change(fill_method=None)
gap_days = p.index.to_series().diff().dt.days
# worst = ret[TICKER].nsmallest(5) ...


## Задание 6. Просадка · дома, 15 минут
Просадка = цена / максимум к этой дате − 1. Максимум берётся только по прошлому: `.cummax()`.

1. Постройте график просадки вашей акции и MCFTR.
2. Найдите максимальную просадку акции: дату пика, дату дна, дату восстановления (если восстановление было). Сколько месяцев длилась просадка?
3. Сравните с MCFTR. Где сейчас ваша акция относительно своего максимума?

**Ответ:** …

In [ ]:
dd = px.dropna() / px.dropna().cummax() - 1
# Дата дна: dd.idxmin(); дата пика: последний максимум до дна; восстановление: первая дата после дна, когда цена >= пика


## Задание 7. Коэффициент Шарпа · 15 минут
Безрисковая дневная доходность – изменение столбца `RUONIA`: `rf = df["RUONIA"].pct_change()`. Избыточная доходность = доходность акции − `rf`. Годовой коэффициент Шарпа = среднее / стандартное отклонение избыточной доходности × √252.

1. Посчитайте коэффициент Шарпа вашей акции и MCFTR за весь период, а также отдельно до 2022 г. и с 2022 г.
2. **Сначала предскажите**, будет ли ваша акция лучше MCFTR в обоих периодах. Затем проверьте.
3. Что оказалось выгоднее за весь период: ваша акция, MCFTR или денежный рынок? Совпадает ли ответ по CAGR и по Шарпу?

**Предсказание:** …

**Ответ:** …

In [ ]:
rf = df["RUONIA"].pct_change()

def sharpe(series, start=None, end=None):
    excess = (series.pct_change(fill_method=None) - rf).loc[start:end].dropna()
    return excess.mean() / excess.std() * np.sqrt(252)

print(f"Шарп MCFTR за весь период: {sharpe(df['MCFTR']):.2f}")
# Посчитайте для своей акции и для двух периодов: sharpe(px, None, "2021-12-31"), sharpe(px, "2022-01-01")


## Задание 8. Проверка кода ИИ-ассистента · дома, 10 минут
Ассистента попросили «посчитать годовую доходность, волатильность, Шарп и максимальную просадку акции». Он написал код ниже. Код работает без ошибок, но числа неверны.

Найдите не меньше трёх ошибок, объясните каждую и исправьте код. Сравните числа до и после исправления.

**Ответ:** …

In [ ]:
# Код ассистента – не меняйте эту ячейку, исправленный вариант напишите ниже
price = df[TICKER + "_close"]
ret = price.pct_change().fillna(0)
annual_return = ret.mean() * 252
annual_vol = ret.std() * 252
sharpe_ai = (annual_return - 0.16) / annual_vol
max_drawdown = (price / price.max() - 1).min()
print(f"{annual_return=:.2%} {annual_vol=:.2%} {sharpe_ai=:.2f} {max_drawdown=:.2%}")

In [ ]:
# Исправленный код


## Задание 9. Вывод · дома, 150–250 слов
Ответьте связным текстом: **стоило ли последние годы держать вашу акцию вместо индекса MCFTR или денежного рынка?**

Опирайтесь на свои числа: CAGR, волатильность, максимальную просадку, коэффициент Шарпа. Сравните результат и с инфляцией: реальная доходность = (1 + CAGR) / (1 + π) − 1, где π – среднегодовая инфляция за период (слайд лекции «Номинальная, реальная и долларовая доходность»). Назовите хотя бы одно ограничение вывода (период, налоги и издержки, выбор акции задним числом и т. п.). Числа без объяснения баллов не дают.

Если пользовались ИИ, запишите: **что спросили**, **что получили**, **как проверили**.

**Вывод:** …

**Журнал ИИ:** …

## Перед сдачей
- Перезапустите ядро и выполните все ячейки сверху вниз.
- Проверьте, что в каждом задании заполнена ячейка «**Ответ:**».
- Сохраните файл `practice03_фамилия.ipynb` и сдайте его способом, который объявит преподаватель.

Непонятные строки кода можно разобрать по [приложению A](https://quantviews.github.io/quant-finance-book/appendix-python.html).